In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
# ==========================================
# 1. Dataset Class
# ==========================================
class OasisDataset(Dataset):
    def __init__(self, csv_file, adj_dir, is_test=False):
        """
        Args:
            csv_file (str): Path to CSV containing node features and target age
            adj_dir (str): Directory containing adjacency matrices as CSVs
            is_test (bool): If True, target y is set to 0
        """
        super(OasisDataset, self).__init__()
        self.df = pd.read_csv(csv_file)
        self.adj_dir = adj_dir
        self.is_test = is_test
        
        # Identify feature columns (Volume and Thickness)
        self.feature_cols = [c for c in self.df.columns if 'volume' in c or 'thickness' in c]
        

    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):
        # 1. Retrieve the row corresponding to the given index 
        row = self.df.iloc[idx]
        sub = row['Subject']        # Subject ID, e.g., "OAS30001" or "Test_Sub_001"
        sess = row['MR_session']    # Session string, e.g., "OAS0001_MR_d0757" or "Sess_01"
        
        # 2. Get target value (age)
        # If this is a test sample, set target to 0 else as is
        if not self.is_test:
            y = torch.tensor([row['age at visit']], dtype=torch.float)
        else:
            y = torch.tensor([0.0], dtype=torch.float)

        # 3. Construct adjacency matrix filename
        # The filename depends on whether this is train/val or test
        # If Train/Val : remove "_MR_" from session to match adjacency file naming e.g., "OAS30001_MR_d0757" -> "OAS30001_d0757.csv"
        # If Test: concatenate sub and sess with '-' e.g., "Test_Sub_001-Sess_01.csv"
        if "Test_Sub" in sub:
            adj_file = f"{sub}-{sess}.csv" # e.g. "Test_Sub_001-Sess_01.csv"
        else:
            sess_clean = sess.replace(f"{sub}_MR_", "")
            adj_file = f"{sub}_{sess_clean}.csv" # e.g. "OAS30001_d0757.csv"
        
        # 4. Load adjacency matrix from CSV
        # index_col=0 ignores row labels
        adj_path = os.path.join(self.adj_dir, adj_file)
        adj_mat = pd.read_csv(adj_path, index_col=0).values # numpy array of shape [num_nodes, num_nodes]
        adj_mat = torch.tensor(adj_mat, dtype=torch.float)
        
        # 5. Prepare node features
        # Original format: 136 features = 68 volume + 68 thickness
        # Reshape into: [68 nodes, 2 features] so that each node has [volume, thickness]
        feat_raw = row[self.feature_cols].astype(float).values    # shape (136, ), first 68 columns are volume while next 68 columns are thickness
        x = torch.tensor(
            feat_raw.reshape(2, 68).T, # First reshape (2, 68) converts feat_raw to [[vol_1, vol_2, ..., vol_68], [thick_1, thick_2, ..., thick_68]], then transpose results each row = one node i.e. [[vol_1, thick_1], [vol_2, thick_2], ..., [vol_68, thick_68]]
            dtype=torch.float
        )                                           # shape [68, 2]

        return {
            "x":x,                      # node features [num_nodes, num_features]
            "adj_mat": adj_mat,         # adjacency matrix [num_nodes, num_nodes]
            "y": y,                     # target scalar
            "name": f"{sub}-{sess}"     # optional identifier
        }

In [3]:
# Paths
base = '/media/bijay-adhikari/Store/BASIRA_GNN/project/git_repo/'
adj_matrices_dir = os.path.join(base, 'data/public/adjacency_matrices/')
train_data_path = os.path.join(base, 'data/public/train_data.csv')
val_data_path = os.path.join(base, 'data/public/val_data.csv')
test_data_path = os.path.join(base, 'data/public/test_data.csv')

# Create Dataset and DataLoaders
train_dataset = OasisDataset(csv_file=train_data_path, adj_dir=adj_matrices_dir, is_test=False)
val_dataset = OasisDataset(csv_file=val_data_path, adj_dir=adj_matrices_dir, is_test=False)
test_dataset = OasisDataset(csv_file=test_data_path, adj_dir=adj_matrices_dir, is_test=True)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Explore dataset
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")


Number of training samples: 353
Number of validation samples: 39
Number of testing samples: 39


In [ ]:
# ==========================================
# 2. Defining SimpleGNN Module
# ==========================================
class SimpleGNN(nn.Module):
    def __init__(self, in_channels=2, hidden_dim=16):
        super(SimpleGNN, self).__init__()
        self.conv1 = nn.Linear(in_channels, hidden_dim)
        self.regressor = nn.Linear(hidden_dim, 1)

    def forward(self, x, adj):
        # Adding self-loops 
        adj = adj + torch.eye(adj.size(0))

        # Message Passing
        # (adj @ x) results in shape [68, in_features]
        agg_feat = torch.matmul(adj, x)

        # Transform and Activate
        x = torch.relu(self.conv1(agg_feat)) # shape [68, hidden_dim]

        # Collapse 68 nodes into 1 vector
        x = torch.mean(x, dim=0)

        # Predict Age
        return self.regressor(x)

In [5]:
# ==========================================
# 3. Training and Validation
# ==========================================
model = SimpleGNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.L1Loss()
n_epoch = 5

for epoch in range(n_epoch):
    # --------- Training ---------------
    model.train()
    total_loss = 0
    for batch in train_loader:
        x = batch['x'].squeeze(0)            # [num_nodes, 2]
        adj = batch['adj_mat'].squeeze(0)    # [num_nodes, num_nodes]
        y = batch['y'].view(-1)                       

        optimizer.zero_grad()
        out = model(x, adj).view(-1)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # -------------- Validation ---------------
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            x = batch['x'].squeeze(0)
            adj = batch['adj_mat'].squeeze(0)
            y = batch['y'].view(-1)

            out = model(x, adj).view(-1)
            loss = criterion(out, y)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}: Train MAE={avg_train_loss:.4f}, Val MAE={avg_val_loss:.4f}")

Epoch 1: Train MAE=112551.0947, Val MAE=27053.1639
Epoch 2: Train MAE=6920.6397, Val MAE=535.1660
Epoch 3: Train MAE=784.3402, Val MAE=255.4527
Epoch 4: Train MAE=201.3769, Val MAE=97.5210
Epoch 5: Train MAE=100.4605, Val MAE=79.5430


In [ ]:
# ==========================================
# 4. Generating Test predictions
# ==========================================
model.eval()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        x = batch['x'].squeeze(0)
        adj = batch['adj_mat'].squeeze(0)

        out = model(x, adj).view(-1)
        subject_session = batch['name'][0]         # get subject_session string e.g., "Test_Sub_001-Sess_01"
        predictions.append({
            'subject_session': subject_session,
            'age_at_visit': out.item()
        })

# Convert to DataFrame
pred_df = pd.DataFrame(predictions, columns=['subject_session', 'age_at_visit'])

# Save to CSV
pred_df.to_csv('predictions.csv', index=False)
print("Predictions saved to predictions.csv")

Predictions saved to predictions.csv


In [7]:
# ==========================================
# 5. Encrypting the prediction file
# ==========================================

In [8]:
!pip install pycryptodome


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
!python /media/bijay-adhikari/Store/BASIRA_GNN/project/git_repo/encryption/encrypt_submission.py \
        --input /media/bijay-adhikari/Store/BASIRA_GNN/project/git_repo/data/baseline_code/predictions.csv \
        --key /media/bijay-adhikari/Store/BASIRA_GNN/project/git_repo/encryption/public_key.pem \
        --output /media/bijay-adhikari/Store/BASIRA_GNN/project/git_repo/submissions/sample_predictions.enc

✅ Success! Encrypted file saved as: /media/bijay-adhikari/Store/BASIRA_GNN/project/git_repo/submissions/sample_predictions.enc
You can now submit this .enc file via Pull Request.
